# 1. Dependencies

## CUDA

In [2]:
import os
import sys

_cuda_root = os.path.join(sys.prefix, "targets", "x86_64-linux")
if os.path.isdir(os.path.join(_cuda_root, "include")):
    os.environ.setdefault("CUDA_PATH", _cuda_root)

%load_ext cuml.accel

cuML: Accelerator installed.


## Common Libraries

In [3]:
import glob
import time
from typing import cast, Any

In [4]:
import json
import joblib
from joblib import Parallel, delayed, parallel_config
from joblib import parallel_config

In [5]:
import math
import cupy as cp
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs

## Plotting

In [6]:
import matplotlib.pyplot as plt

## Pre-Processing

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA
# from imblearn.over_sampling import SMOTE

## Models

### KNN

In [8]:
from cuml.neighbors import KNeighborsClassifier, NearestNeighbors

## Evaluation

In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix
)

## Global Variables

### Path

In [10]:
PATH_CACHE = "cache"

In [11]:
PATH_FOLDER_CSV = "cse-cic-ids2018"
PATH_FOLDER_RAW = "data-raw"

In [12]:
PATH_FOLDER_NO_INF = "data-no-inf"

In [13]:
PATH_FOLDER_SPLIT_DATA = "data-split-feature"
PATH_FOLDER_SPLIT_LABEL = "data-split-label"

In [14]:
PATH_IMPUTER_TRANSFORMER = "median-imputer.json"
PATH_IMPUTER = os.path.join(PATH_CACHE,PATH_IMPUTER_TRANSFORMER)

In [15]:
PATH_FOLDER_IMPUTED = "data-imputed"

In [16]:
PATH_SCALER_TRANSFORMER = "standard-scaler.pkl"
PATH_SCALER = os.path.join(PATH_CACHE,PATH_SCALER_TRANSFORMER)

In [17]:
PATH_FOLDER_SCALED = "data-scaled"

In [18]:
PATH_FOLDER_IPCA_TRANSFORMER = os.path.join(PATH_CACHE,"pca")
PATH_IPCA = "ipca-transformer.pkl"

In [19]:
PATH_FOLDER_IPCA = "data-ipca"

In [20]:
PATH_FOLDER_SMOTE_TRANSFORMER = os.path.join(PATH_CACHE, "smote")
PATH_SMOTE = "smote-transformer.pkl"

In [21]:
PATH_FOLDER_SMOTE = "data-smote"

In [22]:
PATH_FOLDER_MODEL = "trained-model"

In [23]:
PATH_FOLDER_PREDICTION_RESULT = "prediction-result"

### Others

In [24]:
LABEL_COLUMN = "Label"

# Features are persisted to Parquet as float32 instead of float64: half the file
# size, ~2x faster scan_parquet, and every downstream model runs in float32
# anyway. Re-run the scaling / IPCA / SMOTE steps to regenerate existing files.
PARQUET_FLOAT_DTYPE = "float32"

## Helper Functions

In [25]:
def get_all_file_names(source_folder_name:str,file_format:str = ""):
    """List file names in source_folder_name that end with `.{file_format}`, sorted alphabetically.

    Note: if file_format is left empty, this returns an empty list (files are only ever
    appended when a format is given), so a format should always be supplied.
    """
    files = []
    for file in os.listdir(source_folder_name):
        if len(file_format)>0:
            if file.endswith(f".{file_format}"):
                files.append(file)
    return sorted(files)


In [26]:
def filter_file_names(list_file_names:list, filter_word:str):
    """Return only the file names that contain filter_word as a substring (e.g. a date like "2018-02-14")."""
    filtered_file_names = []
    for file_name in list_file_names:
        if filter_word in file_name:
            filtered_file_names.append(file_name)
    return filtered_file_names

In [27]:
def to_pandas_dataframe(data, template_column: list) -> pd.DataFrame:
    """Build a DataFrame from `data` and reindex its columns to match `template_column`."""
    df = pd.DataFrame(data)
    df = df.reindex(columns=template_column)
    return df

In [28]:
def get_split_parquet_files(source_folder_name: str, split_name: str) -> list[str]:
    """Return sorted paths to all Parquet files under `source_folder_name/split_name` (e.g. .../train)."""
    files = sorted(glob.glob(os.path.join(source_folder_name, split_name, "*.parquet")))
    if not files:
        raise FileNotFoundError(f"No Parquet files found in {os.path.join(source_folder_name, split_name)}")
    print(f"Found {len(files)} {split_name} files")
    return files

In [29]:
def get_polars_data_frame_with_label(
        source_folder_name: str, 
        split_name: str = "train", 
        label_column: str = LABEL_COLUMN, 
        downcast_to_float32: bool = True
    ) -> tuple[pl.DataFrame, pl.Series]:

    label_column = label_column.lower()
    files = get_split_parquet_files(source_folder_name, split_name)

    lazy_frame = pl.concat(pl.scan_parquet(file) for file in files)
    if downcast_to_float32:
        lazy_frame = lazy_frame.with_columns(cs.numeric().cast(pl.Float32))

    dataframe = lazy_frame.collect(engine="streaming")
    if label_column not in dataframe.columns:
        raise ValueError(f"Label column '{label_column}' not found in dataframe.")
    x = dataframe.drop(label_column)
    y = dataframe[label_column]
    return x, y

In [30]:
def get_polars_data_frame_without_label(
        source_folder_name: str, 
        split_name: str="", 
        downcast_to_float32: bool = True
    ) -> pl.DataFrame:
    if len(split_name) > 0:
        files = get_split_parquet_files(source_folder_name, split_name)
    else:
        files = get_all_file_names(source_folder_name)
    total = len(files)

    lazy_frames = []
    for i,file in enumerate(files,start=1):
        print(f"Processing [{i}/{total}] {file}...", end="\r", flush=True)
        lazy_frames.append(pl.scan_parquet(file))
    lazy_frame = pl.concat(lazy_frames, how="diagonal_relaxed")

    if downcast_to_float32:
        lazy_frame = lazy_frame.with_columns(cs.numeric().cast(pl.Float32))

    dataframe = lazy_frame.collect(engine="streaming")

    # del lazy_frames, lazy_frame, files, total
    return dataframe

### Model Training

In [31]:
def check_openmp_threads() -> int:
    """Report the thread count that actually governs the KNN search.

    Once sklearn dispatches a brute-force search to `ArgKmin.compute()` it
    threads with OpenMP and ignores `n_jobs` entirely. If this prints 1 the
    search runs single-threaded and will take roughly `n_cores` times longer.
    """
    from sklearn.utils._openmp_helpers import _openmp_effective_n_threads

    n_threads = _openmp_effective_n_threads()
    print(f"OpenMP effective threads: {n_threads}")
    try:
        import threadpoolctl
        for info in threadpoolctl.threadpool_info():
            print(f"  {info['user_api']:>8} / {info['internal_api']:<12}"
                  f" threads={info['num_threads']}")
    except ImportError:
        print("  (pip install threadpoolctl for per-library detail)")
    return n_threads

In [32]:
def dump_trained_model(model,name: str,subfolder: str,folder: str = PATH_FOLDER_MODEL):
    target_folder = os.path.join(folder, subfolder)
    os.makedirs(target_folder, exist_ok=True)
    file_path = os.path.join(target_folder,name)
    joblib.dump(model, file_path)
    return file_path

### Model Evaluation

In [33]:
def evaluate(true,pred) ->pd.DataFrame:
    accuracy = accuracy_score(true,pred)
    precision = precision_score(true,pred)
    recall = recall_score(true,pred)
    f1 = f1_score(true,pred)
    cm = confusion_matrix(true,pred)
    results = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix":cm
    } 
    return pd.DataFrame(results)

# 2. Pre-Processing

## 2.1. Change CSV to Parquet

In [119]:
CHUNK_SIZE = 100000
COLUMNS_TO_DROP = {
    "flow id", "src ip", "source ip", "src port", "source port",
    "dst ip", "destination ip", "timestamp",
}

In [120]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.str.strip().str.lower()
    return df

In [121]:
def drop_unwanted_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=COLUMNS_TO_DROP, errors="ignore")

In [122]:
def dataframe_to_parquet(chunk: pd.DataFrame, target_folder_name: str, file_name: str, template_columns: list[str] | None = None):
    chunk = normalize_columns(chunk)
    chunk = drop_unwanted_columns(chunk)

    if template_columns is not None:
        chunk = chunk.reindex(columns=template_columns)

    label = LABEL_COLUMN.lower()
    for col in chunk.columns:
        if col != label:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("float64")

    chunk = chunk[chunk[label].str.lower() != label]
    chunk = chunk.dropna(subset=[label])
    output_file = os.path.join(target_folder_name, file_name)
    chunk.to_parquet(output_file, engine="pyarrow", compression="snappy", index=False)

In [123]:
def create_template_columns(files: list, source_folder_name: str):
    template_columns = []
    seen = set()
    for file_name in files:
        source_file = os.path.join(source_folder_name, file_name)
        cols = pd.read_csv(source_file, nrows=0).columns.str.strip().str.lower().tolist()
        for c in cols:
            if c not in seen and c not in COLUMNS_TO_DROP:
                seen.add(c)
                template_columns.append(c)
    return template_columns

In [124]:
def get_template_columns(source_folder_name, template_column_path: str = "dataframe-index.pkl"):
    csv_files = get_all_file_names(source_folder_name, "csv")
    template_columns = create_template_columns(csv_files,source_folder_name)
    joblib.dump(template_columns, template_column_path)

In [125]:
def _convert_single_csv_file(
    source_folder_name: str,
    target_folder_name: str,
    file_name: str,
    template_columns: list[str],
    chunk_size: int,
):
    source_file = os.path.join(source_folder_name, file_name)
    base_name = os.path.splitext(file_name)[0]

    for chunk_number, chunk in enumerate(pd.read_csv(source_file, chunksize=chunk_size, low_memory=False), start=1):
        dataframe_to_parquet(chunk, target_folder_name, f"{base_name}_{chunk_number:05d}.parquet", template_columns)

def convert_all_file_to_parquet(
    source_folder_name: str,
    target_folder_name: str,
    chunk_size: int = CHUNK_SIZE
):
    csv_files = get_all_file_names(source_folder_name, "csv")
    template_columns = create_template_columns(csv_files,source_folder_name)

    total = len(csv_files)
    os.makedirs(target_folder_name, exist_ok=True)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_convert_single_csv_file)(source_folder_name, target_folder_name, file_name, template_columns, chunk_size)
            for file_name in csv_files
        )

    print(f"Finished processing {total} files.")

In [126]:
convert_all_file_to_parquet(PATH_FOLDER_CSV, PATH_FOLDER_RAW, CHUNK_SIZE)

FileNotFoundError: [Errno 2] No such file or directory: 'cse-cic-ids2018'

## 2.2 Exploratory Data Analysis (EDA)

## 2.3. Change Infinite Values to NaN


Infinite values ($\infty$ and $-\infty$ / `np.inf` and `-np.inf`) are converted to `NaN` values to ensure that they can be handled consistently during the subsequent missing-value imputation process. This transformation allows both originally missing values and invalid infinite values to be processed using the same imputation method.

In [ ]:
def calculate_inf_values(
    source_folder_name: str
):
    parquet_files = get_all_file_names(source_folder_name,"parquet")
    total = len(parquet_files)
    count = 0
    for i, file_name in enumerate(parquet_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)
        source_file = os.path.join(source_folder_name, file_name)
        df = pd.read_parquet(source_file)
        inf_count = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
        count += inf_count
    print("\n")
    print(f"Completed. Processed {total} files.")
    print(f"Found {count} inf values")

In [ ]:
def _change_inf_to_nan_single_file(source_folder_name: str, target_folder_name: str, file_name: str):
    source_file = os.path.join(source_folder_name, file_name)
    target_file = os.path.join(target_folder_name, file_name)

    df = pd.read_parquet(source_file)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.to_parquet(target_file, index=False)

def change_inf_to_nan(
    source_folder_name: str,
    target_folder_name: str
):
    os.makedirs(target_folder_name, exist_ok=True)

    parquet_files = get_all_file_names(source_folder_name,"parquet")
    total = len(parquet_files)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_change_inf_to_nan_single_file)(source_folder_name, target_folder_name, file_name)
            for file_name in parquet_files
        )

    print(f"Completed. Processed {total} files.")

In [ ]:
change_inf_to_nan(PATH_FOLDER_RAW, PATH_FOLDER_NO_INF)

### Evaluation

The dataset was cleaned by converting both positive infinite  and negative infinite values ($\infty$ and $-\infty$ / `np.inf` and `-np.inf`) to `NaN`. A total of **121,886 infinite values** were identified across **168 Parquet files** before the cleaning process. After the transformation, no infinite values remained in the dataset.

**Before cleaning:**

In [ ]:
calculate_inf_values(PATH_FOLDER_RAW)

**After cleaning:**

In [ ]:
calculate_inf_values(PATH_FOLDER_NO_INF)

This ensures that all infinite values are handled as missing values and can subsequently be processed during the missing-value imputation stage.


## Train/Validation/Test Split

The CSE-CIC-IDS2018 dataset requires careful consideration when dividing the data into training, validation, and test sets. This is because the attack classes are not uniformly distributed across the dataset; instead, specific attack scenarios were conducted on particular dates. As a result, directly splitting the dataset based on individual days may cause some attack classes to be absent from one or more subsets.

The distribution of attack scenarios across the data collection dates is presented below:

| Date  | Attack(s)                           |
|-------|-------------------------------------|
| 14-02 | FTP-BruteForce, SSH-Bruteforce      |
| 15-02 | DoS-GoldenEye, DoS-Slowloris        |
| 16-02 | DoS-SlowHTTPTest, DoS-Hulk          |
| 20-02 | DDoS-LOIC-HTTP, DDoS-LOIC-UDP       |
| 21-02 | DDoS-LOIC-UDP, DDoS-HOIC            |
| 22-02 | Web Brute Force, XSS, SQL Injection |
| 23-02 | Web Brute Force, XSS, SQL Injection |
| 28-02 | Infiltration                        |
| 01-03 | Infiltration                        |
| 02-03 | Bot                                 |

This distribution indicates that several attack classes are associated with only one or a small number of collection dates. Therefore, assigning entire dates directly to the training, validation, or test set could result in certain attack classes being completely absent from the training data. Such a split would make the experiment evaluate unseen attack-class generalization rather than the intended robustness of the ML-IDS against input disturbances.

Therefore, the primary experiment uses a **stratified train/validation/test split based on the attack label**, ensuring that the attack classes are represented across the three subsets. The validation and test sets are kept separate from the training data to prevent information leakage during model development and final evaluation.

A separate day- or scenario-based split may subsequently be used as an additional experiment to evaluate the model's ability to generalize to traffic collected under different attack scenarios.


In [ ]:
bruteforce = "2018-02-14"
dos_golden = "2018-02-15"
dos_hulk = "2018-02-16"
ddos_http = "2018-02-20"
ddos_udp = "2018-02-21"
web_first = "2018-02-22"
web_second = "2018-02-23"
infiltration_first = "2018-02-28"
infiltration_second = "2018-03-01"
botnet = "2018-03-02"

In [ ]:
def create_train_dev_test_folder(source_folder_name: str, features_folder_name: str, labels_folder_name: str, target_day):

    parquet_files = get_all_file_names(source_folder_name, "parquet")
    parquet_files = filter_file_names(parquet_files, target_day)

    if not parquet_files:
        print(f"No Parquet files found for {target_day}")
        raise ValueError(f"No Parquet files found for {target_day}")
    print(f"Found {len(parquet_files)} files for {target_day}")

    split_names = ("train", "dev", "test")
    feature_folders = {}
    label_folders = {}
    for split_name in split_names:
        feature_folder = os.path.join(features_folder_name, split_name)
        label_folder = os.path.join(labels_folder_name, split_name)
        os.makedirs(feature_folder, exist_ok=True)
        os.makedirs(label_folder, exist_ok=True)
        feature_folders[split_name] = feature_folder
        label_folders[split_name] = label_folder

    return parquet_files, feature_folders, label_folders

In [ ]:
def _split_single_parquet_file(
    source_folder_name: str,
    file_name: str,
    feature_folders: dict[str, str],
    label_folders: dict[str, str],
    label_column: str,
    dev_size: float,
    test_size: float,
    random_state: int,
):
    source_file = os.path.join(source_folder_name, file_name)
    df = pd.read_parquet(source_file)

    train_df, temp_df = train_test_split(
        df,
        test_size=dev_size + test_size,
        random_state=random_state,
        stratify=df[label_column],
    )
    dev_df, test_df = train_test_split(
        temp_df,
        test_size=test_size / (dev_size + test_size),
        random_state=random_state,
        stratify=temp_df[label_column],
    )

    for split_name, split_df in (("train", train_df), ("dev", dev_df), ("test", test_df)):
        labels = split_df[[label_column]]
        features = split_df.drop(columns=[label_column])

        features.to_parquet(os.path.join(feature_folders[split_name], file_name), index=False)
        labels.to_parquet(os.path.join(label_folders[split_name], file_name), index=False)

In [ ]:
def split_parquet_files(
    source_folder_name: str,
    parquet_files: list[str],
    feature_folders: dict[str, str],
    label_folders: dict[str, str],
    label_column: str,
    dev_size: float,
    test_size: float,
    random_state: int,
):
    total = len(parquet_files)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_split_single_parquet_file)(
                source_folder_name, file_name, feature_folders, label_folders,
                label_column, dev_size, test_size, random_state,
            )
            for file_name in parquet_files
        )

    print(f"Completed splitting {total} files.")

In [ ]:
def split_a_single_day(
    source_folder_name: str,
    features_folder_name: str,
    labels_folder_name: str,
    target_day: str,
    train_size: float = 0.60,
    dev_size: float = 0.20,
    test_size: float = 0.20,
    label_column: str = LABEL_COLUMN.lower(),
    random_state: int = 42,
):
    if min(train_size, dev_size, test_size) < 0.0:
        raise ValueError("train_size, dev_size, test_size must be non-negative")
    if not math.isclose(train_size + dev_size + test_size, 1.0, abs_tol=1e-9):
        raise ValueError("train_size + dev_size + test_size must equal 1.0")

    parquet_files, feature_folders, label_folders = create_train_dev_test_folder(
        source_folder_name, features_folder_name, labels_folder_name, target_day
    )

    split_parquet_files(
        source_folder_name,
        parquet_files,
        feature_folders,
        label_folders,
        label_column,
        dev_size,
        test_size,
        random_state,
    )

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, bruteforce)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, dos_golden)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, dos_hulk)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, ddos_http)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, ddos_udp)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, web_first)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, web_second)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, infiltration_first)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, infiltration_second)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, botnet)

Files are successfully split.

## Imputation

Missing values are filled using **per-column medians computed from the training split only**, then applied to train/dev/test. Using the median (rather than the mean) avoids distortion from the heavy outliers typical of network flow features (e.g. flow duration, byte counts). Fitting the medians on the training split only and reusing them for dev/test prevents information leakage from the validation/test sets into preprocessing.

### Create Imputation

imputation for NaN

In [ ]:
def get_numeric_columns(combined: pl.DataFrame) -> list[str]:
    schema = combined.collect_schema()

    numeric_cols = []
    for column, dtype in zip(schema.names(), schema.dtypes()):
        if dtype.is_numeric():
            numeric_cols.append(column)

    print(f"Found {len(numeric_cols)} numeric columns")
    return numeric_cols

In [ ]:
def compute_medians(combined: pl.DataFrame, numeric_cols: list[str]) -> pl.DataFrame:
    median_exprs = []
    total = len(numeric_cols)

    for i,column in enumerate(numeric_cols,start=1):
        print(f"Processing [{i}/{total}] {column}...", end="\r", flush=True)
        expr = pl.col(column).median().alias(column)
        median_exprs.append(expr)

    medians = combined.select(median_exprs)
    return medians

In [ ]:
def build_median_dict(medians: pl.DataFrame, numeric_cols: list[str]) -> dict[str, float]:
    median_dict = {}
    null_columns = []

    for c in numeric_cols:
        val = medians[c][0]
        if val is None:
            null_columns.append(c)
            median_dict[c] = 0.0
        else:
            median_dict[c] = float(val)

    if null_columns:
        print(f"Warning: {len(null_columns)} columns had no non-null values, defaulted to 0.0: {null_columns}")

    return median_dict

In [ ]:
def save_median_dict(median_dict: dict[str, float], output_file: str):
    with open(output_file, "w") as file:
        json.dump(median_dict, file, indent=4)
    print(f"Saved medians to: {output_file}")

In [ ]:
def get_median_imputation(source_folder_name: str, output_file: str) -> dict[str, float]:
    df = get_polars_data_frame_without_label(source_folder_name,"train")
    numeric_cols = get_numeric_columns(df)
    medians = compute_medians(df, numeric_cols)
    median_dict = build_median_dict(medians, numeric_cols)
    save_median_dict(median_dict, output_file)
    return median_dict

In [ ]:
median = get_median_imputation(PATH_FOLDER_SPLIT_DATA, PATH_IMPUTER)
print(json.dumps(median, indent=4))

### Impute Training Data

In [ ]:
PATH_FOLDER_IMPUTED = "data-imputed"

In [ ]:
def impute_dataframe(df:pd.DataFrame, medians):
    for column, median in medians.items():
        if column in df.columns:
            df[column] = df[column].fillna(median)
    return df

In [ ]:
def _impute_single_file(source_file: str, output_split_folder: str, medians: dict[str, float]):
    df = pd.read_parquet(source_file)
    df = impute_dataframe(df, medians)
    output_file = os.path.join(output_split_folder, os.path.basename(source_file))
    df.to_parquet(
        output_file,
        engine="pyarrow",
        compression="snappy",
        index=False
    )

def impute_all_split_files(source_folder_name: str, output_folder_name: str, medians_path: str):
    with open(medians_path, "r") as f:
        medians = json.load(f)

    for split_name in ("train", "dev", "test"):
        files = get_split_parquet_files(source_folder_name, split_name)
        total = len(files)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)

        with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_impute_single_file)(file, output_split_folder, medians)
                for file in files
            )

        print(f"Finished imputing {total} {split_name} files.")

In [ ]:
impute_all_split_files(PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_IMPUTED, PATH_IMPUTER)

## Normalized or Feature Scaling

Features are standardized (zero mean, unit variance) using `StandardScaler`. As with imputation, the scaler is fit with `partial_fit` on the **train split only** (processed file-by-file to avoid loading the full ~16M-row dataset into memory at once), then reused to transform train/dev/test. This keeps dev/test statistically unseen during fitting and puts every feature on a comparable scale, which PCA and distance/gradient-based models both depend on.

### Create Scaler

In [ ]:
def fit_scaler_on_files(scaler: StandardScaler, file_paths: list[str]) -> StandardScaler:
    total = len(file_paths)
    for i, file_path in enumerate(file_paths, start=1):
        print(f"Processing [{i}/{total}] {file_path}...", end="\r", flush=True)
        df = pd.read_parquet(file_path)

        if df.empty:
            print(f"Empty file found: {file_path}")
            continue

        scaler.partial_fit(df)
    return scaler

In [ ]:
def create_standard_scaler(source_folder_name: str, split_name: str = "train") -> StandardScaler:
    file_paths = get_split_parquet_files(source_folder_name, split_name)
    scaler = StandardScaler()
    scaler = fit_scaler_on_files(scaler, file_paths)

    if not hasattr(scaler, "mean_"):
        raise ValueError("The scaler could not be fitted because all files were empty.")

    print(" " * 100, end="\r")
    print(f"Successfully fitted scaler using {len(file_paths)} Parquet file(s) from '{split_name}'.")

    return scaler

In [ ]:
def dump_standard_scaler(scaler: StandardScaler,output_path:str):
    joblib.dump(scaler,output_path)

In [ ]:
def get_standard_scaler(source_folder_name:str,output_path:str):
    scaler = create_standard_scaler(source_folder_name)
    dump_standard_scaler(scaler,output_path)

In [ ]:
get_standard_scaler(PATH_FOLDER_IMPUTED, PATH_SCALER)

### Scale Training Data

In [ ]:
def load_standard_scaler(path:str)->StandardScaler:
    return joblib.load(path)

In [ ]:
def _scale_single_file(file_path: str, scaler: StandardScaler, output_split_folder: str):
    df = pd.read_parquet(file_path)
    columns = df.columns.tolist()

    data = scaler.transform(df).astype(PARQUET_FLOAT_DTYPE, copy=False)
    scaled_df = pd.DataFrame(data, columns=columns)

    scaled_df.to_parquet(
        os.path.join(output_split_folder, os.path.basename(file_path)),
        engine="pyarrow",
        compression="snappy",
        index=False
    )

def scale_split_files(source_folder_name: str, scaler_path: str, output_folder_name: str):
    scaler = load_standard_scaler(scaler_path)

    for split_name in ("train", "dev", "test"):
        file_paths = get_split_parquet_files(source_folder_name, split_name)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)
        total = len(file_paths)

        with parallel_config(backend="loky", inner_max_num_threads=1):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_scale_single_file)(file_path, scaler, output_split_folder)
                for file_path in file_paths
            )

        print(f"Finished scaling {total} {split_name} files.")

In [ ]:
scale_split_files(PATH_FOLDER_IMPUTED, PATH_SCALER, PATH_FOLDER_SCALED)

## Principal Component Analysis (PCA)

`IncrementalPCA` is used instead of standard `PCA` because it supports `partial_fit` on mini-batches, so the scaled dataset never needs to be fully loaded into memory. Transformers are fit for a range of component counts (5-40) so the explained-variance-vs-components trade-off can be inspected before committing to a final value.

### Creating Incremental Principal Component Analysis

In [ ]:
LIST_PC_COMPONENTS = [5,10,15,20,25,30,35,40]

In [ ]:
def fit_ipca_on_files(ipca: IncrementalPCA, file_paths: list[str]) -> IncrementalPCA:
    total = len(file_paths)
    for i, file_path in enumerate(file_paths, start=1):
        print(f"Processing [{i}/{total}] {file_path}...", end="\r", flush=True)
        df = pd.read_parquet(file_path)

        if df.empty:
            print(f"Empty file found: {file_path}")
            continue

        ipca.partial_fit(df)
    print(" " * 100, end="\r")
    return ipca

In [ ]:
def create_ipca(n_components,source_folder_name: str, split_name: str = "train") -> IncrementalPCA:
    print(f"Creating IPCA transfromer with {n_components} pc")
    file_paths = get_split_parquet_files(source_folder_name, split_name)
    ipca = IncrementalPCA(n_components=n_components)
    ipca = fit_ipca_on_files(ipca, file_paths)
    print(f"Successfully fitted ipca with {n_components} pc from '{split_name}'.")

    return ipca

In [ ]:
def get_ipca_transformer_path(pc:int, ipca_path:str)->str:
    return f"{pc}-pc-{ipca_path}"

In [ ]:
def dump_ipca(ipca: IncrementalPCA, transformer_folder:str = PATH_FOLDER_IPCA_TRANSFORMER,ipca_path:str = PATH_IPCA):
    filename = get_ipca_transformer_path(ipca.n_components_,ipca_path)
    joblib.dump(ipca,os.path.join(transformer_folder, filename))

In [ ]:
def get_ipca(list_pc_components: list, source_folder_name: str, output_path: str):
    os.makedirs(PATH_FOLDER_IPCA_TRANSFORMER, exist_ok=True)
    for pc in list_pc_components:
        ipca = create_ipca(pc, source_folder_name)
        dump_ipca(ipca, PATH_FOLDER_IPCA_TRANSFORMER, output_path)

In [ ]:
get_ipca(LIST_PC_COMPONENTS,PATH_FOLDER_SCALED, PATH_IPCA)

### Evaluate Incremental Principal Component Analysis

In [ ]:
def load_ipca(pc:int,transformer_folder:str = PATH_FOLDER_IPCA_TRANSFORMER,ipca_path:str = PATH_IPCA) -> IncrementalPCA:
    filename = get_ipca_transformer_path(pc,ipca_path)
    return joblib.load(os.path.join(transformer_folder, filename))

In [ ]:
def evaluate_ipca_variance(list_pc_components: list) -> pd.DataFrame:
    rows = []
    for pc in list_pc_components:
        ipca = load_ipca(pc)

        cum_var = np.cumsum(ipca.explained_variance_ratio_)
        rows.append({
            "n_components": pc,
            "total_explained_variance": cum_var[-1],
            "explained_variance_ratio": ipca.explained_variance_ratio_,
        })

    return pd.DataFrame(rows)

In [ ]:
def plot_ipca_variance(evaluation_results: pd.DataFrame):
    plt.figure(figsize=(8, 5))

    x = evaluation_results["n_components"]
    y = evaluation_results["total_explained_variance"]
    plt.plot(x, y, marker="o", color="black")
    for xi, yi in zip(x, y):
        plt.annotate(f"{yi:.3f}", (xi, yi), xytext=(0, 8), textcoords="offset points", ha="center")

    plt.axhline(0.95, color="red", linestyle="--", label="95% threshold")
    plt.xlabel("Principal Components")
    plt.ylabel("Cumulative Explained Variance")
    plt.title("IPCA: Variance Retained vs. Principal Components")
    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
ipca_variance_results = evaluate_ipca_variance(LIST_PC_COMPONENTS)
plot_ipca_variance(ipca_variance_results)

### Implement Incremental Principal Component Analysis

`IPCA_PC_USED = 25` was chosen from the variance plot above as the smallest evaluated component count that clears the 95% cumulative explained-variance threshold.

In [ ]:
IPCA_PC_USED = 25

In [ ]:
def _ipca_transform_single_file(file_path: str, ipca: IncrementalPCA, output_split_folder: str):
    df = pd.read_parquet(file_path)

    data = ipca.transform(df).astype(PARQUET_FLOAT_DTYPE, copy=False)
    pc_columns = [f"pc{i+1}" for i in range(data.shape[1])]
    scaled_df = pd.DataFrame(data, columns=pc_columns)

    scaled_df.to_parquet(
        os.path.join(output_split_folder, os.path.basename(file_path)),
        engine="pyarrow",
        compression="snappy",
        index=False
    )

def ipca_split_files(source_folder_name: str, ipca_path: str, output_folder_name: str):
    ipca = load_ipca(IPCA_PC_USED)

    for split_name in ("train", "dev", "test"):
        file_paths = get_split_parquet_files(source_folder_name, split_name)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)
        total = len(file_paths)

        with parallel_config(backend="loky", inner_max_num_threads=1):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_ipca_transform_single_file)(file_path, ipca, output_split_folder)
                for file_path in file_paths
            )

        print(f"Finished scaling {total} {split_name} files.")

In [ ]:
ipca_split_files(PATH_FOLDER_SCALED, PATH_IPCA, PATH_FOLDER_IPCA)

## Synthetic Minor Oversampling Technique

CSE-CIC-IDS2018 is heavily imbalanced (Benign traffic dominates; some attack classes are a small fraction of a percent). SMOTE is applied to the **training split only** (never dev/test, so evaluation still reflects real-world class balance) to synthesize minority-class samples and reduce the model's bias toward the majority class.

SMOTE resamples the **PCA-reduced train features** (`PATH_FOLDER_IPCA`, paired with the original `PATH_FOLDER_SPLIT_LABEL` labels), since that's the final feature representation a downstream model would train on. Fitting concatenates every train file via polars (`load_train_features_and_labels`), since imbalanced-learn needs the full split in memory to compute global class counts. Once fitted, the transformer is applied **one file at a time** (`load_smote`, `resample_with_smote`, `save_smote_train_data`) using pandas instead - matching the streaming approach used for IPCA/normalization elsewhere - so resampling never needs the whole split in memory at once; imbalanced-learn only accepts numpy arrays, so the conversion happens right at the `fit_resample` call and the result is wrapped back into a `pd.DataFrame`/`pd.Series`.

Because some attack classes (e.g. SQL Injection) have very few samples in this dataset, `fit_smote` clamps `k_neighbors` down to what the smallest class can support instead of letting `fit_resample` raise on the default `k_neighbors=5`. Fitting is kept separate from resampling: `fit_smote` fits and `dump_smote` persists the fitted transformer via joblib right away, and `resample_with_smote` performs the actual oversampling per file afterwards. Two things can still go wrong at the per-file level that the global fit can't see: (1) a class that's fine globally can be scarce in one specific chunk (e.g. one file has only 5 `Benign` rows against 59,995 `DDOS attack-HOIC` rows), so `resample_with_smote` re-checks `k_neighbors` against that file's own smallest class and clamps further if needed; (2) since each day's capture is chunked in original time order, most individual chunks fall entirely outside the attack window and end up **100% Benign** (123 of the 168 train files, in practice) - `fit_resample` requires at least 2 classes, so `resample_with_smote` detects a single-class file and passes it through unresampled rather than erroring.

### Create SMOTE

In [ ]:
def load_train_features_and_labels(features_folder_name: str, labels_folder_name: str, split_name: str = "train") -> tuple[pl.DataFrame, pl.Series]:
    feature_files = get_split_parquet_files(features_folder_name, split_name)
    label_files = get_split_parquet_files(labels_folder_name, split_name)

    print(f"Loading {len(feature_files)} '{split_name}' feature file(s)...", end="", flush=True)
    features_df = pl.concat([pl.scan_parquet(f) for f in feature_files]).collect()
    labels_df = pl.concat([pl.scan_parquet(f) for f in label_files]).select(LABEL_COLUMN.lower()).collect()

    print(f"Loaded {features_df.height} rows for '{split_name}'.")
    return features_df, labels_df.to_series()

In [ ]:
def get_class_count(labels: pl.Series) -> dict:
    counts = labels.value_counts()
    return dict(zip(counts[labels.name].to_list(), counts["count"].to_list()))

In [ ]:
def fit_smote(features: pl.DataFrame, labels: pl.Series, random_state: int = 42) -> SMOTE:
    class_counts = get_class_count(labels)
    smallest_class_count = min(class_counts.values())
    k_neighbors = max(1, min(5, smallest_class_count - 1))
    
    if k_neighbors < 5:
        print(f"Warning: smallest class has {smallest_class_count} samples; reducing k_neighbors to {k_neighbors}")

    smote = SMOTE(random_state=random_state, k_neighbors=k_neighbors)
    with parallel_config(n_jobs=-1):
        smote.fit(features.to_numpy(), labels.to_numpy())
    return smote

In [ ]:
def dump_smote(smote: SMOTE, transformer_folder: str = PATH_FOLDER_SMOTE_TRANSFORMER, smote_path: str = PATH_SMOTE):
    os.makedirs(transformer_folder, exist_ok=True)
    output_path = os.path.join(transformer_folder, smote_path)
    joblib.dump(smote, output_path)
    print(f"Saved fitted SMOTE transformer to: {output_path}")

In [ ]:
def get_smote_transformer(features_folder_name: str, labels_folder_name: str):
    features, labels = load_train_features_and_labels(features_folder_name, labels_folder_name)
    smote = fit_smote(features, labels)
    dump_smote(smote)

In [ ]:
get_smote_transformer(PATH_FOLDER_IPCA, PATH_FOLDER_SPLIT_LABEL)

### Implement SMOTE

In [ ]:
def load_smote(transformer_folder: str = PATH_FOLDER_SMOTE_TRANSFORMER, smote_path: str = PATH_SMOTE) -> SMOTE:
    return joblib.load(os.path.join(transformer_folder, smote_path))

In [ ]:
def resample_with_smote(smote: SMOTE, features: pd.DataFrame, labels: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    if labels.nunique() < 2:
        return features, labels

    smallest_class_count = labels.value_counts().min()
    k_neighbors = max(1, min(smote.k_neighbors, smallest_class_count - 1))

    if k_neighbors < smote.k_neighbors:
        print(f"Warning: smallest class in this file has {smallest_class_count} samples; reducing k_neighbors to {k_neighbors}")
        smote = SMOTE(random_state=smote.random_state, k_neighbors=k_neighbors, sampling_strategy=smote.sampling_strategy)

    features_resampled, labels_resampled = cast(
        tuple[np.ndarray, np.ndarray],
        smote.fit_resample(features.to_numpy(), labels.to_numpy()),
    )
    resampled_features = pd.DataFrame(
        features_resampled, columns=features.columns
    ).astype(PARQUET_FLOAT_DTYPE)
    return resampled_features, pd.Series(labels_resampled, name=labels.name)

In [ ]:
def save_smote_train_data(features: pd.DataFrame, labels: pd.Series, file_name: str, output_folder_name: str = PATH_FOLDER_SMOTE):
    output_split_folder = os.path.join(output_folder_name, "train")
    os.makedirs(output_split_folder, exist_ok=True)

    output_df = features.assign(**{LABEL_COLUMN.lower(): labels.to_numpy()})
    output_df.to_parquet(
        os.path.join(output_split_folder, file_name),
        engine="pyarrow",
        compression="snappy",
        index=False
    )

In [ ]:
def _smote_resample_single_file(feature_file_path: str, label_file_path: str, smote: SMOTE, output_folder_name: str):
    features = pd.read_parquet(feature_file_path)
    labels = pd.read_parquet(label_file_path)[LABEL_COLUMN.lower()]

    features_resampled, labels_resampled = resample_with_smote(smote, features, labels)
    save_smote_train_data(features_resampled, labels_resampled, os.path.basename(feature_file_path), output_folder_name)

def smote_resample_files(features_folder_name: str, labels_folder_name: str, output_folder_name: str = PATH_FOLDER_SMOTE):
    smote = load_smote()

    feature_files = get_split_parquet_files(features_folder_name, "train")
    label_files = get_split_parquet_files(labels_folder_name, "train")
    total = len(feature_files)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_smote_resample_single_file)(feature_file_path, label_file_path, smote, output_folder_name)
            for feature_file_path, label_file_path in zip(feature_files, label_files)
        )

    print(f"Finished SMOTE-resampling {total} train file(s).")

In [ ]:
smote_resample_files(PATH_FOLDER_IPCA, PATH_FOLDER_SPLIT_LABEL, PATH_FOLDER_SMOTE)

### Evaluate SMOTE

In [ ]:
def get_class_count_from_label(source_folder_path:str): 
    label_files = get_split_parquet_files(source_folder_path, "train")
    labels = pl.concat([pl.scan_parquet(f) for f in label_files]).select(LABEL_COLUMN.lower()).collect()
    return get_class_count(labels.to_series()), len(labels)

In [ ]:
before_class_count,before_row_count = get_class_count_from_label(PATH_FOLDER_SPLIT_LABEL)
print(f"Row count: {before_row_count}")
print(f"Class count:\n{json.dumps(before_class_count, indent=4)}")

In [ ]:
after_class_count,after_row_count = get_class_count_from_label(PATH_FOLDER_SMOTE)
print(f"Row count: {after_row_count}")
print(f"Class count:\n{json.dumps(after_class_count, indent=4)}")

In [ ]:
def plot_smote_comparison(before_class_count: dict[str, int], after_class_count: dict[str, int],) -> None:
    classes = sorted(set(before_class_count) | set(after_class_count))
    before = []
    for class_name in classes:
        before.append(before_class_count.get(class_name, 0))

    after = []
    for class_name in classes:
        after.append(after_class_count.get(class_name, 0))

    df = pd.DataFrame({"Class": classes, "Before SMOTE": before, "After SMOTE": after,})

    ax = df.set_index("Class").plot(
        kind="bar",
        figsize=(14, 7),
        width=0.8,
    )

    ax.set_title("Class Distribution Before and After SMOTE")
    ax.set_xlabel("Class")
    ax.set_ylabel("Number of Samples")

    plt.xticks(rotation=45, ha="right")
    plt.legend(title="Dataset")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_smote_comparison(before_class_count,after_class_count)

# 3. Load Datasets

## Load Training Data

In [34]:
train_x, train_y = get_polars_data_frame_with_label(PATH_FOLDER_SMOTE)

Found 168 train files


## Load Validation Data

In [35]:
dev_x = get_polars_data_frame_without_label(PATH_FOLDER_IPCA,"dev")
dev_y = get_polars_data_frame_without_label(PATH_FOLDER_SPLIT_LABEL,"dev").to_series()

Found 168 dev files
Found 168 dev files] data-ipca/dev/2018-03-02-Friday_TrafficForML_CICFlowMeter_00011.parquet......


## Load Test Data

In [ ]:
test_x = get_polars_data_frame_without_label(PATH_FOLDER_IPCA,"test")
test_y = get_polars_data_frame_without_label(PATH_FOLDER_SPLIT_LABEL,"test").to_series()

Found 168 test files
Found 168 test files data-ipca/test/2018-03-02-Friday_TrafficForML_CICFlowMeter_00011.parquet......


# 4. K-Nearest Neighbor

In [36]:
KNN_QUERY_CHUNK = 100_000

In [37]:
def get_knn_name(n,metric,weights):
    return f"knn-k-{n}-m-{metric}-w-{weights}.pkl"

In [38]:
def train_knn(train_x,train_y,n_components,distance_metric,weighting) -> KNeighborsClassifier:
    cuml = KNeighborsClassifier(n_components,metric=distance_metric,weights=weighting)
    cuml.fit(train_x,train_y)
    return cuml

## Hyperparameter Search for KNN

Uses nearest neighbors

In [39]:
list_of_n_components = [i for i in range(1,15,2)]
list_of_distance_metrics = ["manhattan","euclidean","cosine"]
list_of_weighing = ["uniform","distance"]

In [40]:
def create_nearest_neighbors(train_x, list_of_n_components, list_of_distance_metrics) -> list:
    models = []
    biggest_k = max(list_of_n_components)
    for distance_metric in list_of_distance_metrics:
        nn = NearestNeighbors(
            n_neighbors=biggest_k,
            metric=distance_metric,
            algorithm="brute",
            output_type="cupy",
        )
        nn.fit(train_x)
        models.append({"metric": distance_metric, "model": nn})
    return models

In [41]:
def _vote(neighbor_labels, distances, weighing, classes):
    if weighing == "uniform":
        weights = cp.ones_like(distances, dtype=cp.float32)
    else:
        weights = 1.0 / distances
        exact = cp.isinf(weights)
        rows = exact.any(axis=1)
        weights[rows] = exact[rows].astype(weights.dtype)

    votes = cp.stack(
        [cp.where(neighbor_labels == c, weights, 0.0).sum(axis=1) for c in classes.tolist()],
        axis=1,
    )
    return classes[votes.argmax(axis=1)]


In [42]:
def _scores(y_true, y_pred, classes):
    acc = float((y_true == y_pred).mean())
    f1_sum = 0.0
    for c in classes.tolist():
        tp = int(((y_pred == c) & (y_true == c)).sum())
        fp = int(((y_pred == c) & (y_true != c)).sum())
        fn = int(((y_pred != c) & (y_true == c)).sum())
        denom = 2 * tp + fp + fn
        f1_sum += 0.0 if denom == 0 else 2 * tp / denom
    return acc, f1_sum / len(classes)

In [43]:
def calculate_results(models, train_y, test_x, test_y,
                      list_of_n_components, list_of_weighing) -> pd.DataFrame:
    train_y = cp.asarray(train_y)
    test_y = cp.asarray(test_y)
    classes = cp.unique(train_y)
    biggest_k = max(list_of_n_components)

    rows = []
    for entry in models:
        distances, indices = entry["model"].kneighbors(test_x, n_neighbors=biggest_k)
        neighbor_labels = train_y[indices]

        for k in list_of_n_components:
            d, lab = distances[:, :k], neighbor_labels[:, :k]
            for weighing in list_of_weighing:
                pred = _vote(lab, d, weighing, classes)
                acc, f1 = _scores(test_y, pred, classes)
                rows.append({
                    "metric": entry["metric"], "k": k, "weighing": weighing,
                    "accuracy": acc, "f1_macro": f1,
                })

    return pd.DataFrame(rows).sort_values("f1_macro", ascending=False, ignore_index=True)

In [44]:
def hyperparameter_search(train_x, train_y, valid_x, valid_y,
                          list_of_n_components,
                          list_of_distance_metrics,
                          list_of_weighing,
                          verbose=True):
    n_combos = (len(list_of_n_components)
                * len(list_of_distance_metrics)
                * len(list_of_weighing))
    if verbose:
        print(f"Searching {n_combos} combinations "
              f"({len(list_of_distance_metrics)} index builds)...")

    models = create_nearest_neighbors(
        train_x, list_of_n_components, list_of_distance_metrics
    )

    results = calculate_results(
        models, train_y, valid_x, valid_y,
        list_of_n_components, list_of_weighing
    )

    best = results.iloc[0].to_dict()
    if verbose:
        print(f"Best: metric={best['metric']}, k={int(best['k'])}, "
              f"weighing={best['weighing']} "
              f"| f1_macro={best['f1_macro']:.4f}, acc={best['accuracy']:.4f}")

    return results, best

In [ ]:
results, best = hyperparameter_search(
    train_x, train_y, dev_x, dev_y,
    list_of_n_components, list_of_distance_metrics, list_of_weighing,
)
results.head(10)

Searching 42 combinations (3 index builds)...
